# FUNCTIONS

## summarize_mpf_results

In [1]:
from pathlib import Path
import pandas as pd
import re
import json


def summarize_mpf_results(results_dir="results"):
    """
    Create a summary table of all MPF walk-forward experiments.

    Each experiment is identified primarily by its JSON metadata file.

    Status:
        RUNNING    -> JSON exists but result CSV does not yet exist
        COMPLETED  -> JSON and result CSV both exist

    Run duration:
        COMPLETED -> CSV modification time - JSON start time
        RUNNING   -> current time - JSON start time

    Evaluation:
        classical_ranking -> *_uniq_classical_ranking.csv exists
        idr_ranking       -> *_uniq_idr_ranking.csv exists
        final_selection   -> *_uniq_final_selection.csv exists

    Parameters
    ----------
    results_dir : str or Path
        Directory containing MPF result files.

    Returns
    -------
    pd.DataFrame
        Summary table, one row per experiment/run.
    """

    results_dir = Path(results_dir)

    # ------------------------------------------------------------------
    # Find JSON files.
    #
    # JSON is the primary source because it is created when the
    # experiment starts.
    #
    # Evaluation JSON files containing "_uniq_" are excluded.
    # ------------------------------------------------------------------

    json_files = sorted(
        (
            path
            for path in results_dir.glob("wf_*.json")
            if "_uniq_" not in path.name
        ),
        key=lambda x: x.stat().st_mtime,
        reverse=True
    )

    rows = []

    # ------------------------------------------------------------------
    # Filename pattern
    # ------------------------------------------------------------------

    pattern = re.compile(
        r"wf_(?P<strategy>.+?)"
        r"_cal(?P<cal>\d+)"
        r"_move(?P<move>\d+)"
        r"_test(?P<test>\d+)"
        r"_ncomp(?P<ncomp_start>\d+)-(?P<ncomp_end>\d+)"
        r"_(?P<timestamp>\d{8}_\d{6})_"
        r"(?P<run_id>[a-f0-9]+)\.json"
    )

    # ------------------------------------------------------------------
    # Current time
    # ------------------------------------------------------------------

    now = pd.Timestamp.now()

    # ------------------------------------------------------------------
    # Process every JSON file
    # ------------------------------------------------------------------

    for json_file in json_files:

        match = pattern.match(json_file.name)

        if not match:
            continue

        info = match.groupdict()

        # ------------------------------------------------------------------
        # Component range
        # ------------------------------------------------------------------

        ncomp_start = int(info["ncomp_start"])
        ncomp_end = int(info["ncomp_end"])

        if ncomp_start == ncomp_end:
            components = str(ncomp_start)
        else:
            components = f"{ncomp_start}–{ncomp_end}"

        # ------------------------------------------------------------------
        # Read JSON metadata
        # ------------------------------------------------------------------

        metadata = {}

        try:
            with open(json_file, "r") as f:
                metadata = json.load(f)

        except Exception as e:
            print(
                f"Warning: could not read "
                f"{json_file.name}: {e}"
            )

        # ------------------------------------------------------------------
        # Start time
        #
        # Prefer timestamp_utc from JSON.
        # Fall back to filename timestamp.
        # ------------------------------------------------------------------

        try:
            start_time = pd.to_datetime(
                metadata.get(
                    "timestamp_utc",
                    info["timestamp"]
                ),
                format="%Y%m%d_%H%M%S"
            )

        except Exception:

            start_time = pd.to_datetime(
                info["timestamp"],
                format="%Y%m%d_%H%M%S"
            )

        # ------------------------------------------------------------------
        # Result CSV
        # ------------------------------------------------------------------

        stem = json_file.stem

        result_file = (
            results_dir / f"{stem}.csv"
        )

        # ------------------------------------------------------------------
        # New evaluation output files
        # ------------------------------------------------------------------

        classical_file = (
            results_dir
            / f"{stem}_uniq_classical_ranking.csv"
        )

        idr_file = (
            results_dir
            / f"{stem}_uniq_idr_ranking.csv"
        )

        final_selection_file = (
            results_dir
            / f"{stem}_uniq_final_selection.csv"
        )

        # ------------------------------------------------------------------
        # Determine status and end time
        # ------------------------------------------------------------------

        if result_file.exists():

            status = "COMPLETED"

            # CSV modification time = completion time
            end_time = pd.to_datetime(
                result_file.stat().st_mtime,
                unit="s"
            )

        else:

            status = "RUNNING"

            # No CSV yet -> use current time
            end_time = now

        # ------------------------------------------------------------------
        # Duration
        # ------------------------------------------------------------------

        duration = end_time - start_time

        duration_seconds = int(
            duration.total_seconds()
        )

        if duration_seconds >= 0:

            hours, remainder = divmod(
                duration_seconds,
                3600
            )

            minutes, seconds = divmod(
                remainder,
                60
            )

            duration_str = (
                f"{hours:02d}:"
                f"{minutes:02d}:"
                f"{seconds:02d}"
            )

        else:

            duration_str = "ERROR"

        # ------------------------------------------------------------------
        # Read result CSV if available
        # ------------------------------------------------------------------

        result_df = pd.DataFrame()

        if result_file.exists():

            try:

                result_df = pd.read_csv(
                    result_file
                )

                n_rows = len(result_df)

            except Exception as e:

                print(
                    f"Warning: could not read "
                    f"{result_file.name}: {e}"
                )

                n_rows = None

        else:

            n_rows = None

        # ------------------------------------------------------------------
        # Determine OOS end date
        # ------------------------------------------------------------------

        end_date = None

        if not result_df.empty:

            for column in [
                "test_end",
                "TEST_END",
                "end_date",
                "END_DATE"
            ]:

                if column in result_df.columns:

                    dates = pd.to_datetime(
                        result_df[column],
                        errors="coerce"
                    )

                    if dates.notna().any():

                        end_date = dates.max().date()

                    break

        # ------------------------------------------------------------------
        # New evaluation status
        #
        # Each evaluation result is independent.
        #
        # This means a partially evaluated experiment can be detected,
        # e.g. classical=True, IDR=True, final=False.
        # ------------------------------------------------------------------

        classical_ranking = (
            classical_file.exists()
        )

        idr_ranking = (
            idr_file.exists()
        )

        final_selection = (
            final_selection_file.exists()
        )

        # ------------------------------------------------------------------
        # Add row
        # ------------------------------------------------------------------

        rows.append({

            # --------------------------------------------------------------
            # Run status
            # --------------------------------------------------------------

            "status": status,

            "run_date": start_time.date(),

            "start_time": start_time.strftime(
                "%H:%M:%S"
            ),

            "end_time": (
                end_time.strftime("%H:%M:%S")
                if status == "COMPLETED"
                else None
            ),

            "duration": duration_str,

            "run_id": info["run_id"],

            # --------------------------------------------------------------
            # Experiment
            # --------------------------------------------------------------

            "strategy": metadata.get(
                "STRATEGY",
                info["strategy"]
            ),

            "start_date": metadata.get(
                "START_DATE"
            ),

            "end_date": end_date,

            # --------------------------------------------------------------
            # Walk-forward parameters
            # --------------------------------------------------------------

            "calibration": metadata.get(
                "CALIBRATION_WINDOW",
                int(info["cal"])
            ),

            "move": metadata.get(
                "MOVING_PARAM",
                int(info["move"])
            ),

            "test": metadata.get(
                "TEST_SPAN",
                int(info["test"])
            ),

            "components": components,

            # --------------------------------------------------------------
            # Simulations
            # --------------------------------------------------------------

            "stage1_simulations": metadata.get(
                "N_SIMULATIONS_STAGE1"
            ),

            "stage2_simulations": metadata.get(
                "N_SIMULATIONS_STAGE2"
            ),

            # --------------------------------------------------------------
            # Optimization
            # --------------------------------------------------------------

            "threshold_step": metadata.get(
                "THRESHOLD_STEP"
            ),

            "random_state": metadata.get(
                "RANDOM_STATE"
            ),

            # --------------------------------------------------------------
            # Portfolio parameters
            # --------------------------------------------------------------

            "initial_capital": metadata.get(
                "INITIAL_CAPITAL"
            ),

            "cost_per_trade": metadata.get(
                "ABS_COST_FOR_A_TRADE"
            ),

            "percent_cost": metadata.get(
                "PERCENT_COST_FOR_A_TRADE"
            ),

            "max_investment_pct": metadata.get(
                "MAX_INVESTMENT_SIZE_IN_PERCENT"
            ),

            "min_cash_pct": metadata.get(
                "MIN_CASH_IN_PERCENT"
            ),

            # --------------------------------------------------------------
            # Computing
            # --------------------------------------------------------------

            "max_workers": metadata.get(
                "MAX_WORKERS"
            ),

            # --------------------------------------------------------------
            # Evaluation outputs
            # --------------------------------------------------------------

            "classical_ranking": classical_ranking,

            "idr_ranking": idr_ranking,

            "final_selection": final_selection,

            # --------------------------------------------------------------
            # Output information
            # --------------------------------------------------------------

            "result_rows": n_rows,

            "json": json_file.exists(),

            "result_file": (
                result_file.name
                if result_file.exists()
                else None
            ),
        })

    # ----------------------------------------------------------------------
    # Create DataFrame
    # ----------------------------------------------------------------------

    summary = pd.DataFrame(rows)

    if summary.empty:

        print(
            "No MPF experiment JSON files found."
        )

        return summary

    # ----------------------------------------------------------------------
    # Convert dates
    # ----------------------------------------------------------------------

    summary["start_date"] = pd.to_datetime(
        summary["start_date"],
        errors="coerce"
    ).dt.date

    summary["end_date"] = pd.to_datetime(
        summary["end_date"],
        errors="coerce"
    ).dt.date

    # ----------------------------------------------------------------------
    # Sort newest runs first
    # ----------------------------------------------------------------------

    summary = summary.sort_values(
        ["run_date", "start_time"],
        ascending=False
    ).reset_index(drop=True)

    return summary

## plot_selection_stability

In [2]:
from pathlib import Path

import json
import re

import pandas as pd
import matplotlib.pyplot as plt


def plot_selection_stability(
    results_dir="results",
    strategy=None,
    data_start=None,
    data_end=None
):
    """
    Plot selection stability across MPF Stage-2 simulation levels.

    The function dynamically discovers completed MPF experiments from
    their primary JSON metadata files.

    Workflow
    --------
    JSON metadata
        -> main experiment CSV
        -> actual OOS end date
        -> final-selection CSV
        -> Stage-2 simulation level
        -> stability summary
        -> article-quality PDF

    Panels
    ------
    (a) Threshold instability
    (b) OOS performance instability
    (c) Selected portfolio size

    Parameters
    ----------
    results_dir : str or Path
        Directory containing MPF result files.

    strategy : str, optional
        Strategy to plot, e.g. "mean_reversion" or "momentum".
        If None, all strategies found are processed separately.

    data_start : str, optional
        Experiment START_DATE, e.g. "2009-01-01".
        If None, all start dates are considered.

    data_end : str, optional
        Actual final OOS date, e.g. "2023-12-29".
        If None, all end dates are considered.

    Returns
    -------
    dict
        One entry per strategy containing:
            "strategy"
            "data_start"
            "data_end"
            "table"
            "pdf"
    """

    results_dir = Path(results_dir)

    # ==============================================================
    # Find primary experiment JSON files
    # ==============================================================

    json_files = sorted(
        (
            path
            for path in results_dir.glob("wf_*.json")
            if "_uniq_" not in path.name
        ),
        key=lambda x: x.stat().st_mtime
    )

    if not json_files:
        print("No MPF experiment JSON files found.")
        return {}

    # ==============================================================
    # Experiment filename pattern
    # ==============================================================

    pattern = re.compile(
        r"wf_(?P<strategy>.+?)"
        r"_cal(?P<cal>\d+)"
        r"_move(?P<move>\d+)"
        r"_test(?P<test>\d+)"
        r"_ncomp(?P<ncomp_start>\d+)-(?P<ncomp_end>\d+)"
        r"_(?P<timestamp>\d{8}_\d{6})_"
        r"(?P<run_id>[a-f0-9]+)\.json"
    )

    experiments = []

    # ==============================================================
    # Read experiments
    # ==============================================================

    for json_path in json_files:

        match = pattern.match(json_path.name)

        if not match:
            continue

        info = match.groupdict()

        # ----------------------------------------------------------
        # Read metadata
        # ----------------------------------------------------------

        try:
            with open(json_path, "r") as f:
                metadata = json.load(f)
        except Exception as e:
            print(
                f"Warning: could not read "
                f"{json_path.name}: {e}"
            )
            continue

        # ----------------------------------------------------------
        # Strategy
        # ----------------------------------------------------------

        experiment_strategy = metadata.get(
            "STRATEGY",
            info["strategy"]
        )

        if (
            strategy is not None
            and experiment_strategy != strategy
        ):
            continue

        # ----------------------------------------------------------
        # Experiment start date
        # ----------------------------------------------------------

        experiment_start = metadata.get(
            "START_DATE"
        )

        if experiment_start is None:
            continue

        experiment_start = pd.Timestamp(
            experiment_start
        ).strftime("%Y-%m-%d")

        if (
            data_start is not None
            and experiment_start != str(data_start)
        ):
            continue

        # ----------------------------------------------------------
        # Main experiment CSV
        #
        # Same convention used by summarize_mpf_results().
        # ----------------------------------------------------------

        stem = json_path.stem

        result_file = (
            results_dir / f"{stem}.csv"
        )

        # Fallback to output_file from metadata if necessary.

        if not result_file.exists():

            output_file = metadata.get(
                "output_file"
            )

            if output_file:

                candidate = (
                    results_dir
                    / Path(output_file).name
                )

                if candidate.exists():
                    result_file = candidate

        if not result_file.exists():
            continue

        # ----------------------------------------------------------
        # Read main experiment CSV
        # ----------------------------------------------------------

        try:
            result_df = pd.read_csv(
                result_file
            )
        except Exception as e:
            print(
                f"Warning: could not read "
                f"{result_file.name}: {e}"
            )
            continue

        if result_df.empty:
            continue

        # ----------------------------------------------------------
        # Determine actual OOS end date
        # ----------------------------------------------------------

        experiment_end = None

        for column in [
            "test_end",
            "TEST_END",
            "end_date",
            "END_DATE"
        ]:

            if column in result_df.columns:

                dates = pd.to_datetime(
                    result_df[column],
                    errors="coerce"
                )

                if dates.notna().any():
                    experiment_end = (
                        dates.max().strftime("%Y-%m-%d")
                    )

                break

        if experiment_end is None:
            print(
                f"Warning: could not determine OOS end "
                f"for {result_file.name}"
            )
            continue

        # ----------------------------------------------------------
        # Optional end-date filter
        # ----------------------------------------------------------

        if (
            data_end is not None
            and experiment_end != str(data_end)
        ):
            continue

        # ----------------------------------------------------------
        # Stage-2 simulations
        # ----------------------------------------------------------

        stage2 = metadata.get(
            "N_SIMULATIONS_STAGE2"
        )

        if stage2 is None:
            continue

        stage2 = int(stage2)

        # ----------------------------------------------------------
        # Final selection file
        # ----------------------------------------------------------

        final_selection_file = (
            results_dir
            / f"{stem}_uniq_final_selection.csv"
        )

        if not final_selection_file.exists():
            continue

        try:
            selection_df = pd.read_csv(
                final_selection_file
            )
        except Exception as e:
            print(
                f"Warning: could not read "
                f"{final_selection_file.name}: {e}"
            )
            continue

        required_columns = [
            "selection_method",
            "threshold_instability",
            "oos_instability",
            "max_num_components"
        ]

        if not all(
            column in selection_df.columns
            for column in required_columns
        ):
            print(
                f"Warning: required columns missing in "
                f"{final_selection_file.name}"
            )
            continue

        experiments.append({
            "strategy": experiment_strategy,
            "data_start": experiment_start,
            "data_end": experiment_end,
            "stage2": stage2,
            "selection": selection_df.copy(),
            "run_id": metadata.get(
                "run_id",
                info["run_id"]
            )
        })

    # ==============================================================
    # No matching experiments
    # ==============================================================

    if not experiments:
        print(
            "No matching completed experiments found."
        )
        return {}

    # ==============================================================
    # Group by strategy and actual data period
    # ==============================================================

    grouped = {}

    for experiment in experiments:

        key = (
            experiment["strategy"],
            experiment["data_start"],
            experiment["data_end"]
        )

        grouped.setdefault(
            key,
            []
        ).append(experiment)

    outputs = {}

    # ==============================================================
    # Process each strategy / period
    # ==============================================================

    for key, runs in grouped.items():

        (
            experiment_strategy,
            experiment_start,
            experiment_end
        ) = key

        rows = []

        # ----------------------------------------------------------
        # Build stability table
        # ----------------------------------------------------------

        for run in runs:

            df = run["selection"].copy()

            df["selection_method"] = (
                df["selection_method"]
                .astype(str)
                .str.strip()
            )

            grouped_selection = (
                df.groupby(
                    "selection_method",
                    as_index=False
                )
                .agg(
                    threshold_instability=(
                        "threshold_instability",
                        "mean"
                    ),
                    oos_instability=(
                        "oos_instability",
                        "mean"
                    ),
                    selected_portfolio_size=(
                        "max_num_components",
                        "mean"
                    )
                )
            )

            for _, row in grouped_selection.iterrows():

                method = row[
                    "selection_method"
                ]

                method_lower = method.lower()

                if (
                    "idr" in method_lower
                    or "instability" in method_lower
                ):
                    method_label = (
                        "Instability–Degradation"
                    )

                elif "classical" in method_lower:
                    method_label = "Classical"

                else:
                    method_label = method

                rows.append({
                    "stage2_simulations": run["stage2"],
                    "selection_method": method_label,
                    "threshold_instability": (
                        row["threshold_instability"]
                    ),
                    "oos_instability": (
                        row["oos_instability"]
                    ),
                    "selected_portfolio_size": (
                        row["selected_portfolio_size"]
                    )
                })

        table = pd.DataFrame(rows)

        if table.empty:
            continue

        # ----------------------------------------------------------
        # Average duplicate runs for the same Stage-2 level
        # ----------------------------------------------------------

        table = (
            table.groupby(
                [
                    "stage2_simulations",
                    "selection_method"
                ],
                as_index=False
            )
            .mean(numeric_only=True)
            .sort_values(
                [
                    "stage2_simulations",
                    "selection_method"
                ]
            )
            .reset_index(drop=True)
        )

        # ----------------------------------------------------------
        # Keep the two article methods
        # ----------------------------------------------------------

        preferred_methods = [
            "Classical",
            "Instability–Degradation"
        ]

        table = table[
            table["selection_method"].isin(
                preferred_methods
            )
        ].copy()

        if table.empty:
            continue

        # ==========================================================
        # Figure
        # ==========================================================

        fig, axes = plt.subplots(
            1,
            3,
            figsize=(15, 4.8)
        )

        x_values = sorted(
            table["stage2_simulations"].unique()
        )

        styles = {
            "Classical": {
                "linestyle": "-",
                "marker": "o"
            },
            "Instability–Degradation": {
                "linestyle": "--",
                "marker": "s"
            }
        }

        panels = [
            (
                "threshold_instability",
                "Threshold instability",
                "(a)"
            ),
            (
                "oos_instability",
                "OOS performance instability",
                "(b)"
            ),
            (
                "selected_portfolio_size",
                "Selected portfolio size",
                "(c)"
            )
        ]

        # ----------------------------------------------------------
        # Plot panels
        # ----------------------------------------------------------

        for panel_number, (
            ax,
            (column, ylabel, panel_label)
        ) in enumerate(
            zip(axes, panels)
        ):

            for method in preferred_methods:

                subset = table[
                    table["selection_method"] == method
                ].sort_values(
                    "stage2_simulations"
                )

                if subset.empty:
                    continue

                style = styles[method]

                # Smaller markers for instability panels.
                if panel_number < 2:
                    marker_size = 3.5
                else:
                    marker_size = 5

                ax.plot(
                    subset["stage2_simulations"],
                    subset[column],
                    linestyle=style["linestyle"],
                    marker=style["marker"],
                    color="black",
                    linewidth=1.5,
                    markersize=marker_size,
                    markerfacecolor="white",
                    markeredgecolor="black",
                    markeredgewidth=1.0,
                    zorder=3,
                    label=method
                )

            ax.set_xlabel(
                "Stage-2 simulations"
            )

            ax.set_ylabel(
                ylabel
            )

            ax.set_xticks(
                x_values
            )

            ax.grid(
                True,
                linestyle=":",
                linewidth=0.8,
                alpha=0.7,
                zorder=0
            )

            ax.text(
                0.0,
                1.04,
                panel_label,
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontweight="bold"
            )

        # ==========================================================
        # Common legend
        # ==========================================================

        handles, labels = (
            axes[0].get_legend_handles_labels()
        )

        if handles:

            fig.legend(
                handles,
                labels,
                loc="upper center",
                bbox_to_anchor=(0.5, 1.02),
                ncol=2,
                frameon=False
            )

        # ==========================================================
        # Figure title
        # ==========================================================

        fig.suptitle(
            (
                f"Selection stability: "
                f"{experiment_strategy.replace('_', ' ').title()} "
                f"({experiment_start} to {experiment_end})"
            ),
            y=1.08,
            fontsize=12
        )

        fig.tight_layout()

        # ==========================================================
        # Save PDF
        # ==========================================================

        pdf_name = (
            "article_selection_stability_"
            f"{experiment_strategy}_"
            f"{experiment_start}_"
            f"{experiment_end}.pdf"
        )

        pdf_path = (
            results_dir / pdf_name
        )

        fig.savefig(
            pdf_path,
            format="pdf",
            bbox_inches="tight"
        )

        plt.close(fig)

        outputs[
            experiment_strategy
        ] = {
            "strategy": experiment_strategy,
            "data_start": experiment_start,
            "data_end": experiment_end,
            "table": table,
            "pdf": pdf_path
        }

        print(
            f"Created: {pdf_path.name}"
        )

                # ==========================================================
        # Print results table
        # ==========================================================

        print("\n" + "=" * 110)
        print(
            "SELECTION STABILITY RESULTS — "
            f"{experiment_strategy.replace('_', ' ').title()} "
            f"({experiment_start} to {experiment_end})"
        )
        print("=" * 110)

        print(
            table.to_string(
                index=False,
                formatters={
                    "threshold_instability":
                        lambda x: f"{x:.6f}",
                    "oos_instability":
                        lambda x: f"{x:.6f}",
                    "selected_portfolio_size":
                        lambda x: f"{x:.2f}",
                },
            )
        )

        print("=" * 110)

    return outputs

## plot_classical_performance_comparison

In [3]:
from pathlib import Path
import json
import re
import pandas as pd
import matplotlib.pyplot as plt


def plot_classical_performance_comparison(
    results_dir="results",
    strategy=None,
    data_start=None,
    data_end=None
):
    """
    Compare Classical and IDR selection on the four classical
    performance measures across MPF Stage-2 simulation levels.

    Measures
    --------
    (a) Positive OOS proportion
    (b) Mean excess return
    (c) Compound excess return
    (d) Sharpe of excess return

    The function dynamically discovers completed MPF experiments
    from their primary JSON metadata files.

    Workflow
    --------
    JSON metadata
        -> main experiment CSV
        -> actual OOS end date
        -> classical ranking CSV
        -> IDR ranking CSV
        -> selected configuration
        -> performance comparison
        -> article-quality PDF

    Parameters
    ----------
    results_dir : str or Path
        Directory containing MPF result files.

    strategy : str, optional
        Strategy to plot, e.g. "mean_reversion" or "momentum".
        If None, all strategies found are processed separately.

    data_start : str, optional
        Experiment START_DATE.

    data_end : str, optional
        Actual final OOS date.

    Returns
    -------
    dict
        One entry per strategy containing:
            "strategy"
            "data_start"
            "data_end"
            "table"
            "pdf"
    """

    results_dir = Path(results_dir)

    # ============================================================
    # Find primary experiment JSON files
    # ============================================================

    json_files = sorted(
        (
            path
            for path in results_dir.glob("wf_*.json")
            if "_uniq_" not in path.name
        ),
        key=lambda x: x.stat().st_mtime
    )

    if not json_files:
        print("No MPF experiment JSON files found.")
        return {}

    # ============================================================
    # Experiment filename pattern
    # ============================================================

    pattern = re.compile(
        r"wf_(?P<strategy>.+?)"
        r"_cal(?P<cal>\d+)"
        r"_move(?P<move>\d+)"
        r"_test(?P<test>\d+)"
        r"_ncomp(?P<ncomp_start>\d+)-(?P<ncomp_end>\d+)"
        r"_(?P<timestamp>\d{8}_\d{6})_"
        r"(?P<run_id>[a-f0-9]+)\.json"
    )

    experiments = []

    # ============================================================
    # Read experiments
    # ============================================================

    for json_path in json_files:

        match = pattern.match(json_path.name)

        if not match:
            continue

        info = match.groupdict()

        # --------------------------------------------------------
        # Read metadata
        # --------------------------------------------------------

        try:
            with open(json_path, "r", encoding="utf-8") as f:
                metadata = json.load(f)
        except Exception as e:
            print(
                f"Warning: could not read "
                f"{json_path.name}: {e}"
            )
            continue

        # --------------------------------------------------------
        # Strategy
        # --------------------------------------------------------

        experiment_strategy = metadata.get(
            "STRATEGY",
            info["strategy"]
        )

        if (
            strategy is not None
            and experiment_strategy != strategy
        ):
            continue

        # --------------------------------------------------------
        # Experiment start date
        # --------------------------------------------------------

        experiment_start = metadata.get("START_DATE")

        if experiment_start is None:
            continue

        experiment_start = pd.Timestamp(
            experiment_start
        ).strftime("%Y-%m-%d")

        if (
            data_start is not None
            and experiment_start != str(data_start)
        ):
            continue

        # --------------------------------------------------------
        # Main experiment CSV
        # --------------------------------------------------------

        stem = json_path.stem

        result_file = (
            results_dir / f"{stem}.csv"
        )

        if not result_file.exists():

            output_file = metadata.get(
                "output_file"
            )

            if output_file:

                candidate = (
                    results_dir
                    / Path(output_file).name
                )

                if candidate.exists():
                    result_file = candidate

        if not result_file.exists():
            continue

        # --------------------------------------------------------
        # Read main experiment CSV
        # --------------------------------------------------------

        try:
            result_df = pd.read_csv(result_file)
        except Exception as e:
            print(
                f"Warning: could not read "
                f"{result_file.name}: {e}"
            )
            continue

        if result_df.empty:
            continue

        # --------------------------------------------------------
        # Determine actual OOS end date
        # --------------------------------------------------------

        experiment_end = None

        for column in [
            "test_end",
            "TEST_END",
            "end_date",
            "END_DATE"
        ]:

            if column in result_df.columns:

                dates = pd.to_datetime(
                    result_df[column],
                    errors="coerce"
                )

                if dates.notna().any():

                    experiment_end = (
                        dates.max().strftime("%Y-%m-%d")
                    )

                break

        if experiment_end is None:

            print(
                f"Warning: could not determine OOS end "
                f"for {result_file.name}"
            )

            continue

        # --------------------------------------------------------
        # Optional end-date filter
        # --------------------------------------------------------

        if (
            data_end is not None
            and experiment_end != str(data_end)
        ):
            continue

        # --------------------------------------------------------
        # Stage-2 simulations
        # --------------------------------------------------------

        stage2 = metadata.get(
            "N_SIMULATIONS_STAGE2"
        )

        if stage2 is None:
            continue

        stage2 = int(stage2)

        # --------------------------------------------------------
        # Ranking files
        # --------------------------------------------------------

        classical_file = (
            results_dir
            / f"{stem}_uniq_classical_ranking.csv"
        )

        idr_file = (
            results_dir
            / f"{stem}_uniq_idr_ranking.csv"
        )

        if (
            not classical_file.exists()
            or not idr_file.exists()
        ):
            continue

        # --------------------------------------------------------
        # Read ranking files
        # --------------------------------------------------------

        try:
            classical_df = pd.read_csv(
                classical_file
            )

            idr_df = pd.read_csv(
                idr_file
            )

        except Exception as e:

            print(
                f"Warning: could not read ranking files "
                f"for {stem}: {e}"
            )

            continue

        # --------------------------------------------------------
        # Required columns
        # --------------------------------------------------------

        required_columns = [
            "max_num_components",
            "P_positive",
            "mean_excess_return",
            "compound_excess_return",
            "sharpe_excess"
        ]

        if not all(
            column in classical_df.columns
            for column in required_columns
        ):
            print(
                f"Warning: required columns missing in "
                f"{classical_file.name}"
            )
            continue

        if not all(
            column in idr_df.columns
            for column in required_columns
        ):
            print(
                f"Warning: required columns missing in "
                f"{idr_file.name}"
            )
            continue

        # --------------------------------------------------------
        # Select the top-ranked configuration
        #
        # Classical:
        # classical_rank == 1
        #
        # IDR:
        # idr_rank == 1
        # --------------------------------------------------------

        if "classical_rank" in classical_df.columns:

            classical_selected = classical_df[
                classical_df["classical_rank"] == 1
            ].copy()

        else:

            classical_selected = (
                classical_df
                .sort_values("classical_average_rank")
                .head(1)
                .copy()
            )

        if "idr_rank" in idr_df.columns:

            idr_selected = idr_df[
                idr_df["idr_rank"] == 1
            ].copy()

        else:

            idr_selected = (
                idr_df
                .sort_values("idr_score")
                .head(1)
                .copy()
            )

        if classical_selected.empty:
            continue

        if idr_selected.empty:
            continue

        # --------------------------------------------------------
        # If ties exist, average the tied selected configurations.
        # --------------------------------------------------------

        classical_values = (
            classical_selected[
                required_columns
            ].mean(numeric_only=True)
        )

        idr_values = (
            idr_selected[
                required_columns
            ].mean(numeric_only=True)
        )

        experiments.append({

            "strategy": experiment_strategy,

            "data_start": experiment_start,

            "data_end": experiment_end,

            "stage2": stage2,

            "classical": classical_values,

            "idr": idr_values,

            "run_id": metadata.get(
                "run_id",
                info["run_id"]
            )
        })

    # ============================================================
    # No matching experiments
    # ============================================================

    if not experiments:

        print(
            "No matching completed experiments found."
        )

        return {}

    # ============================================================
    # Group by strategy and actual data period
    # ============================================================

    grouped = {}

    for experiment in experiments:

        key = (
            experiment["strategy"],
            experiment["data_start"],
            experiment["data_end"]
        )

        grouped.setdefault(
            key,
            []
        ).append(experiment)

    outputs = {}

    # ============================================================
    # Process each strategy / period
    # ============================================================

    for key, runs in grouped.items():

        (
            experiment_strategy,
            experiment_start,
            experiment_end
        ) = key

        rows = []

        # --------------------------------------------------------
        # Build performance table
        # --------------------------------------------------------

        for run in runs:

            for method, values in [
                ("Classical", run["classical"]),
                ("Instability–Degradation", run["idr"])
            ]:

                rows.append({

                    "stage2_simulations":
                        run["stage2"],

                    "selection_method":
                        method,

                    "P_positive":
                        values["P_positive"],

                    "mean_excess_return":
                        values["mean_excess_return"],

                    "compound_excess_return":
                        values["compound_excess_return"],

                    "sharpe_excess":
                        values["sharpe_excess"],

                    "selected_portfolio_size":
                        values["max_num_components"]

                })

        table = pd.DataFrame(rows)

        if table.empty:
            continue

        # --------------------------------------------------------
        # Average duplicate runs
        # --------------------------------------------------------

        table = (
            table.groupby(
                [
                    "stage2_simulations",
                    "selection_method"
                ],
                as_index=False
            )
            .mean(numeric_only=True)
            .sort_values(
                [
                    "stage2_simulations",
                    "selection_method"
                ]
            )
            .reset_index(drop=True)
        )

        # ========================================================
        # Figure
        # ========================================================

        fig, axes = plt.subplots(
            1,
            4,
            figsize=(17, 4.8)
        )

        x_values = sorted(
            table[
                "stage2_simulations"
            ].unique()
        )

        styles = {

            "Classical": {
                "linestyle": "-",
                "marker": "o"
            },

            "Instability–Degradation": {
                "linestyle": "--",
                "marker": "s"
            }

        }

        panels = [

            (
                "P_positive",
                "Positive OOS proportion",
                "(a)"
            ),

            (
                "mean_excess_return",
                "Mean excess return",
                "(b)"
            ),

            (
                "compound_excess_return",
                "Compound excess return",
                "(c)"
            ),

            (
                "sharpe_excess",
                "Sharpe of excess return",
                "(d)"
            )

        ]

        # --------------------------------------------------------
        # Plot panels
        # --------------------------------------------------------

        for panel_number, (
            ax,
            (column, ylabel, panel_label)
        ) in enumerate(
            zip(axes, panels)
        ):

            for method in [
                "Classical",
                "Instability–Degradation"
            ]:

                subset = table[
                    table["selection_method"] == method
                ].sort_values(
                    "stage2_simulations"
                )

                if subset.empty:
                    continue

                style = styles[method]

                ax.plot(
                    subset[
                        "stage2_simulations"
                    ],
                    subset[column],

                    linestyle=style["linestyle"],

                    marker=style["marker"],

                    color="black",

                    linewidth=1.5,

                    markersize=4,

                    markerfacecolor="white",

                    markeredgecolor="black",

                    markeredgewidth=1.0,

                    zorder=3,

                    label=method
                )

            ax.set_xlabel(
                "Stage-2 simulations"
            )

            ax.set_ylabel(
                ylabel
            )

            ax.set_xticks(
                x_values
            )

            ax.grid(
                True,
                linestyle=":",
                linewidth=0.8,
                alpha=0.7,
                zorder=0
            )

            ax.text(
                0.0,
                1.04,
                panel_label,
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontweight="bold"
            )

        # ========================================================
        # Common legend
        # ========================================================

        handles, labels = (
            axes[0].get_legend_handles_labels()
        )

        if handles:

            fig.legend(
                handles,
                labels,
                loc="upper center",
                bbox_to_anchor=(0.5, 1.02),
                ncol=2,
                frameon=False
            )

        # ========================================================
        # Figure title
        # ========================================================

        fig.suptitle(
            (
                f"Classical performance comparison: "
                f"{experiment_strategy.replace('_', ' ').title()} "
                f"({experiment_start} to {experiment_end})"
            ),
            y=1.08,
            fontsize=12
        )

        fig.tight_layout()

        # ========================================================
        # Save PDF
        # ========================================================

        pdf_name = (
            "article_classical_performance_"
            f"{experiment_strategy}_"
            f"{experiment_start}_"
            f"{experiment_end}.pdf"
        )

        pdf_path = (
            results_dir / pdf_name
        )

        fig.savefig(
            pdf_path,
            format="pdf",
            bbox_inches="tight"
        )

        plt.close(fig)

        outputs[
            experiment_strategy
        ] = {

            "strategy":
                experiment_strategy,

            "data_start":
                experiment_start,

            "data_end":
                experiment_end,

            "table":
                table,

            "pdf":
                pdf_path

        }

        print(
            f"Created: {pdf_path.name}"
        )

                # ==========================================================
        # Print results table
        # ==========================================================

        print("\n" + "=" * 120)
        print(
            "CLASSICAL PERFORMANCE COMPARISON RESULTS — "
            f"{experiment_strategy.replace('_', ' ').title()} "
            f"({experiment_start} to {experiment_end})"
        )
        print("=" * 120)

        print(
            table.to_string(
                index=False,
                formatters={
                    "P_positive":
                        lambda x: f"{x:.6f}",
                    "mean_excess_return":
                        lambda x: f"{x:.6f}",
                    "compound_excess_return":
                        lambda x: f"{x:.6f}",
                    "sharpe_excess":
                        lambda x: f"{x:.6f}",
                    "selected_portfolio_size":
                        lambda x: f"{x:.2f}",
                },
            )
        )

        print("=" * 120)

    return outputs

## plot_performance_stability_tradeoff

In [4]:
def plot_performance_stability_tradeoff(
    results_dir="../results",
    strategy="mean_reversion",
    data_start="2009-01-01",
    data_end="2023-12-29",
    output_file=None,
):
    """
    Plot the empirical performance–stability trade-off between
    Classical and Instability–Degradation (IDR) selection.

    Panel (a):
        Compound benchmark-relative excess return vs
        threshold instability.

    Panel (b):
        Compound benchmark-relative excess return vs
        OOS performance instability.

    Panel (c):
        Compound benchmark-relative excess return vs
        calibration-to-OOS degradation.

    Each Stage-2 simulation level is represented by a pair of
    Classical and IDR observations. Only observations belonging
    to the same Stage-2 level are connected.

    The figure is deliberately kept simple for article use.
    """

    import json
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    results_dir = Path(results_dir)

    if output_file is None:
        output_file = (
            results_dir
            / f"performance_stability_tradeoff_"
              f"{strategy}_{data_start}_{data_end}.pdf"
        )

    # ============================================================
    # Find ranking files
    # ============================================================

    classical_files = sorted(
        results_dir.glob(
            f"wf_{strategy}_*_uniq_classical_ranking.csv"
        )
    )

    idr_files = sorted(
        results_dir.glob(
            f"wf_{strategy}_*_uniq_idr_ranking.csv"
        )
    )

    if not classical_files:
        raise ValueError(
            f"No Classical ranking files found in {results_dir}"
        )

    if not idr_files:
        raise ValueError(
            f"No IDR ranking files found in {results_dir}"
        )

    print(
        f"Found {len(classical_files)} Classical ranking files "
        f"and {len(idr_files)} IDR ranking files."
    )

    # ============================================================
    # Match IDR files by experiment prefix
    # ============================================================

    idr_by_prefix = {}

    for f in idr_files:
        prefix = f.name.replace(
            "_uniq_idr_ranking.csv",
            "",
        )
        idr_by_prefix[prefix] = f

    records = []

    # ============================================================
    # Read experiments
    # ============================================================

    for classical_file in classical_files:

        prefix = classical_file.name.replace(
            "_uniq_classical_ranking.csv",
            "",
        )

        idr_file = idr_by_prefix.get(prefix)

        if idr_file is None:
            continue

        # --------------------------------------------------------
        # Read Stage-2 metadata
        # --------------------------------------------------------

        stage2 = None

        for json_file in results_dir.glob(
            f"{prefix}*.json"
        ):

            try:
                with open(
                    json_file,
                    "r",
                    encoding="utf-8",
                ) as f:
                    meta = json.load(f)

                stage2 = meta.get(
                    "N_SIMULATIONS_STAGE2",
                    meta.get(
                        "stage2_simulations",
                        meta.get("N_SIM_STAGE2"),
                    ),
                )

                if stage2 is not None:
                    break

            except Exception:
                continue

        if stage2 is None:
            continue

        try:
            stage2 = int(stage2)
        except Exception:
            continue

        # --------------------------------------------------------
        # Read ranking files
        # --------------------------------------------------------

        classical = pd.read_csv(
            classical_file
        )

        idr = pd.read_csv(
            idr_file
        )

        required = [
            "compound_excess_return",
            "threshold_instability",
            "oos_instability",
            "degradation",
        ]

        if not all(
            col in classical.columns
            for col in required
        ):
            continue

        if not all(
            col in idr.columns
            for col in required
        ):
            continue

        if "classical_rank" not in classical.columns:
            continue

        if "idr_rank" not in idr.columns:
            continue

        # --------------------------------------------------------
        # Best Classical configuration
        # --------------------------------------------------------

        classical_rank = pd.to_numeric(
            classical["classical_rank"],
            errors="coerce",
        )

        classical_best = classical[
            np.isclose(
                classical_rank,
                1.0,
                equal_nan=False,
            )
        ]

        if classical_best.empty:
            continue

        classical_values = (
            classical_best[required]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .mean()
        )

        # --------------------------------------------------------
        # Best IDR configuration
        # --------------------------------------------------------

        idr_rank = pd.to_numeric(
            idr["idr_rank"],
            errors="coerce",
        )

        idr_best = idr[
            np.isclose(
                idr_rank,
                1.0,
                equal_nan=False,
            )
        ]

        if idr_best.empty:
            continue

        idr_values = (
            idr_best[required]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .mean()
        )

        # --------------------------------------------------------
        # Store
        # --------------------------------------------------------

        records.append(
            {
                "stage2_simulations": stage2,

                "classical_compound_return":
                    classical_values[
                        "compound_excess_return"
                    ],

                "idr_compound_return":
                    idr_values[
                        "compound_excess_return"
                    ],

                "classical_threshold_instability":
                    classical_values[
                        "threshold_instability"
                    ],

                "idr_threshold_instability":
                    idr_values[
                        "threshold_instability"
                    ],

                "classical_oos_instability":
                    classical_values[
                        "oos_instability"
                    ],

                "idr_oos_instability":
                    idr_values[
                        "oos_instability"
                    ],

                "classical_degradation":
                    classical_values[
                        "degradation"
                    ],

                "idr_degradation":
                    idr_values[
                        "degradation"
                    ],
            }
        )

    # ============================================================
    # Check results
    # ============================================================

    if not records:
        raise ValueError(
            "No matching Classical/IDR ranking pairs were found."
        )

    df = pd.DataFrame(records)

    df = (
        df.groupby(
            "stage2_simulations",
            as_index=False,
        )
        .mean(numeric_only=True)
        .sort_values(
            "stage2_simulations"
        )
        .reset_index(drop=True)
    )

    # ============================================================
    # Article-style black-and-white figure
    # ============================================================

    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "axes.titlesize": 10,
            "axes.labelsize": 9,
            "xtick.labelsize": 8,
            "ytick.labelsize": 8,
        }
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 4.8),
    )

    # ============================================================
    # Common drawing function
    # ============================================================

    def draw_panel(
        ax,
        classical_x,
        idr_x,
        classical_y,
        idr_y,
        xlabel,
        ylabel,
        title,
    ):

        # --------------------------------------------------------
        # Draw pairwise connections first
        # --------------------------------------------------------

        for _, row in df.iterrows():

            ax.plot(
                [
                    row[classical_x],
                    row[idr_x],
                ],
                [
                    row[classical_y],
                    row[idr_y],
                ],
                color="black",
                linestyle=":",
                linewidth=0.8,
                zorder=1,
            )

        # --------------------------------------------------------
        # Classical
        # --------------------------------------------------------

        ax.scatter(
            df[classical_x],
            df[classical_y],
            marker="o",
            facecolors="white",
            edgecolors="black",
            s=48,
            linewidths=1.1,
            zorder=3,
            label="Classical",
        )

        # --------------------------------------------------------
        # IDR
        # --------------------------------------------------------

        ax.scatter(
            df[idr_x],
            df[idr_y],
            marker="s",
            facecolors="black",
            edgecolors="black",
            s=42,
            linewidths=1.0,
            zorder=3,
            label="IDR",
        )

        # --------------------------------------------------------
        # Stage-2 labels
        # --------------------------------------------------------

        for _, row in df.iterrows():

            x1 = row[classical_x]
            y1 = row[classical_y]

            x2 = row[idr_x]
            y2 = row[idr_y]

            xm = (x1 + x2) / 2
            ym = (y1 + y2) / 2

            ax.annotate(
                str(
                    int(
                        row["stage2_simulations"]
                    )
                ),
                (xm, ym),
                xytext=(0, 5),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=7.5,
                bbox=dict(
                    facecolor="white",
                    edgecolor="none",
                    pad=0.5,
                ),
                zorder=4,
            )

        # --------------------------------------------------------
        # Labels
        # --------------------------------------------------------

        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title)

        # --------------------------------------------------------
        # Grid
        # --------------------------------------------------------

        ax.grid(
            True,
            linestyle=":",
            linewidth=0.5,
            alpha=0.6,
        )

        # --------------------------------------------------------
        # Direction annotation
        # --------------------------------------------------------

        ax.annotate(
            "higher performance →",
            xy=(0.98, 0.03),
            xycoords="axes fraction",
            ha="right",
            va="bottom",
            fontsize=7.5,
        )

    # ============================================================
    # Panel A — threshold instability
    # ============================================================

    draw_panel(
        axes[0],
        "classical_compound_return",
        "idr_compound_return",
        "classical_threshold_instability",
        "idr_threshold_instability",
        "Compound benchmark-relative excess return",
        "Threshold instability",
        "(a) Performance–threshold stability",
    )

    # ============================================================
    # Panel B — OOS instability
    # ============================================================

    draw_panel(
        axes[1],
        "classical_compound_return",
        "idr_compound_return",
        "classical_oos_instability",
        "idr_oos_instability",
        "Compound benchmark-relative excess return",
        "OOS performance instability",
        "(b) Performance–OOS stability",
    )

    # ============================================================
    # Panel C — calibration-to-OOS degradation
    # ============================================================

    draw_panel(
        axes[2],
        "classical_compound_return",
        "idr_compound_return",
        "classical_degradation",
        "idr_degradation",
        "Compound benchmark-relative excess return",
        "Calibration-to-OOS degradation",
        "(c) Performance–degradation",
    )

    # ============================================================
    # Common legend
    # ============================================================

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=2,
        frameon=False,
    )

    # ============================================================
    # Overall title
    # ============================================================

    fig.suptitle(
        f"Performance–Stability Trade-off: "
        f"{strategy.replace('_', ' ').title()}",
        fontsize=12,
        y=1.06,
    )

    fig.tight_layout(
        rect=[0, 0, 1, 0.94]
    )

    # ============================================================
    # Save
    # ============================================================

    fig.savefig(
        output_file,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)

    print(
        f"Saved: {output_file}"
    )

    # ============================================================
    # Print results table
    # ============================================================

    print("\n" + "=" * 140)
    print("PERFORMANCE–STABILITY–DEGRADATION TRADE-OFF RESULTS")
    print("=" * 140)

    print(
        df.to_string(
            index=False,
            formatters={
                "classical_compound_return":
                    lambda x: f"{x:.6f}",
                "idr_compound_return":
                    lambda x: f"{x:.6f}",
                "classical_threshold_instability":
                    lambda x: f"{x:.6f}",
                "idr_threshold_instability":
                    lambda x: f"{x:.6f}",
                "classical_oos_instability":
                    lambda x: f"{x:.6f}",
                "idr_oos_instability":
                    lambda x: f"{x:.6f}",
                "classical_degradation":
                    lambda x: f"{x:.6f}",
                "idr_degradation":
                    lambda x: f"{x:.6f}",
            },
        )
    )

    print("=" * 140)

    return df

## plot_oos_performance_instability_stacked

In [5]:
def plot_oos_performance_instability_stacked(
    results_dir="../results",
    strategy="mean_reversion",
    data_start="2009-01-01",
    data_end="2023-12-29",
    output_file=None,
):
    """
    Plot OOS performance and OOS instability as one stacked bar
    for each Stage-2 simulation level.

    Structure
    ---------
    X-axis:
        Stage-2 simulations (n_mc2)

    Y-axis:
        Four stacked methods:
            Sharpe
            Calmar
            Return/Risk
            Robust

    For every method:
        black      = OOS performance
        white ///// = OOS instability

    Classical and IDR are displayed in separate panels.

    The method names are taken directly from the 'method' column
    of the Classical and IDR ranking CSV files.

    OOS performance:
        oos_return_mean

    OOS instability:
        oos_instability
    """

    import json
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    # ============================================================
    # Paths
    # ============================================================

    results_dir = Path(results_dir)

    if output_file is None:
        output_file = (
            results_dir
            / f"oos_performance_instability_stacked_"
              f"{strategy}_{data_start}_{data_end}.pdf"
        )

    # ============================================================
    # Find files
    # ============================================================

    classical_files = sorted(
        results_dir.glob(
            f"wf_{strategy}_*_uniq_classical_ranking.csv"
        )
    )

    idr_files = sorted(
        results_dir.glob(
            f"wf_{strategy}_*_uniq_idr_ranking.csv"
        )
    )

    if not classical_files:
        raise ValueError(
            f"No Classical ranking files found in {results_dir}"
        )

    if not idr_files:
        raise ValueError(
            f"No IDR ranking files found in {results_dir}"
        )

    print(
        f"Found {len(classical_files)} Classical ranking files "
        f"and {len(idr_files)} IDR ranking files."
    )

    # ============================================================
    # Match IDR files to Classical files
    # ============================================================

    idr_by_prefix = {}

    for f in idr_files:

        prefix = f.name.replace(
            "_uniq_idr_ranking.csv",
            "",
        )

        idr_by_prefix[prefix] = f

    # ============================================================
    # Read all experiments
    # ============================================================

    records = []

    for classical_file in classical_files:

        prefix = classical_file.name.replace(
            "_uniq_classical_ranking.csv",
            "",
        )

        idr_file = idr_by_prefix.get(prefix)

        if idr_file is None:
            continue

        # --------------------------------------------------------
        # Stage-2 simulations
        # --------------------------------------------------------

        stage2 = None

        for json_file in results_dir.glob(
            f"{prefix}*.json"
        ):

            try:

                with open(
                    json_file,
                    "r",
                    encoding="utf-8",
                ) as f:

                    meta = json.load(f)

                stage2 = meta.get(
                    "N_SIMULATIONS_STAGE2",
                    meta.get(
                        "stage2_simulations",
                        meta.get(
                            "N_SIM_STAGE2"
                        ),
                    ),
                )

                if stage2 is not None:
                    break

            except Exception:
                continue

        if stage2 is None:
            continue

        try:
            stage2 = int(stage2)
        except Exception:
            continue

        # --------------------------------------------------------
        # Read CSVs
        # --------------------------------------------------------

        classical = pd.read_csv(
            classical_file
        )

        idr = pd.read_csv(
            idr_file
        )

        # --------------------------------------------------------
        # Required columns
        # --------------------------------------------------------

        required = [
            "method",
            "oos_return_mean",
            "oos_instability",
        ]

        if not all(
            col in classical.columns
            for col in required
        ):
            continue

        if not all(
            col in idr.columns
            for col in required
        ):
            continue

        # --------------------------------------------------------
        # Store Classical
        # --------------------------------------------------------

        for _, row in classical.iterrows():

            records.append(
                {
                    "stage2_simulations": stage2,
                    "selection": "Classical",
                    "method": row["method"],
                    "oos_performance": pd.to_numeric(
                        row["oos_return_mean"],
                        errors="coerce",
                    ),
                    "oos_instability": pd.to_numeric(
                        row["oos_instability"],
                        errors="coerce",
                    ),
                }
            )

        # --------------------------------------------------------
        # Store IDR
        # --------------------------------------------------------

        for _, row in idr.iterrows():

            records.append(
                {
                    "stage2_simulations": stage2,
                    "selection": "IDR",
                    "method": row["method"],
                    "oos_performance": pd.to_numeric(
                        row["oos_return_mean"],
                        errors="coerce",
                    ),
                    "oos_instability": pd.to_numeric(
                        row["oos_instability"],
                        errors="coerce",
                    ),
                }
            )

    # ============================================================
    # Data frame
    # ============================================================

    df = pd.DataFrame(records)

    if df.empty:
        raise ValueError(
            "No valid Classical/IDR observations found."
        )

    df = df.dropna(
        subset=[
            "oos_performance",
            "oos_instability",
        ]
    )

    # ============================================================
    # Inspect available methods
    # ============================================================

    print(
        "\nMethods found:"
    )

    print(
        sorted(
            df["method"]
            .astype(str)
            .unique()
            .tolist()
        )
    )

    # ============================================================
    # Map method names
    #
    # The actual strings in 'method' may differ slightly, so
    # identify them from their names.
    # ============================================================

    def classify_method(value):

        s = str(value).lower()

        if "sharpe" in s:
            return "Sharpe"

        if "calmar" in s:
            return "Calmar"

        if (
            "return/risk" in s
            or "return_risk" in s
            or "return risk" in s
        ):
            return "Return/Risk"

        if "robust" in s:
            return "Robust"

        return None

    df["metric"] = df["method"].apply(
        classify_method
    )

    # ============================================================
    # Check methods
    # ============================================================

    missing = [
        metric
        for metric in [
            "Sharpe",
            "Calmar",
            "Return/Risk",
            "Robust",
        ]
        if metric not in df["metric"].dropna().unique()
    ]

    if missing:

        print(
            "\nCould not identify:"
        )

        for metric in missing:
            print(
                f"  {metric}"
            )

        print(
            "\nActual method values:"
        )

        print(
            df["method"]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Could not identify all four methods. "
            "Check the 'method' values printed above."
        )

    df = df.dropna(
        subset=["metric"]
    )

    # ============================================================
    # Four metrics in the requested stacking order
    # ============================================================

    metric_order = [
        "Sharpe",
        "Calmar",
        "Return/Risk",
        "Robust",
    ]

    # ============================================================
    # If there are duplicate rows for the same
    # n_mc2 / selection / metric, average them.
    # ============================================================

    df = (
        df.groupby(
            [
                "stage2_simulations",
                "selection",
                "metric",
            ],
            as_index=False,
        )
        [
            [
                "oos_performance",
                "oos_instability",
            ]
        ]
        .mean()
    )

    # ============================================================
    # n_mc2 values
    # ============================================================

    stage2_values = sorted(
        df["stage2_simulations"]
        .unique()
    )

    # ============================================================
    # Plot
    # ============================================================

    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "axes.titlesize": 11,
            "axes.labelsize": 9,
            "xtick.labelsize": 8,
            "ytick.labelsize": 8,
        }
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 6),
        sharey=True,
    )

    # ============================================================
    # Draw one panel
    # ============================================================

    def draw_panel(
        ax,
        selection,
        title,
    ):

        # --------------------------------------------------------
        # Width of each n_mc2 bar
        # --------------------------------------------------------

        bar_width = 0.62

        # --------------------------------------------------------
        # For every n_mc2
        # --------------------------------------------------------

        for i, stage2 in enumerate(
            stage2_values
        ):

            bottom = 0.0

            # ----------------------------------------------------
            # Four metrics are stacked vertically
            # ----------------------------------------------------

            for metric in metric_order:

                row = df[
                    (
                        df[
                            "stage2_simulations"
                        ]
                        == stage2
                    )
                    & (
                        df["selection"]
                        == selection
                    )
                    & (
                        df["metric"]
                        == metric
                    )
                ]

                if row.empty:
                    continue

                performance = float(
                    row[
                        "oos_performance"
                    ].iloc[0]
                )

                instability = float(
                    row[
                        "oos_instability"
                    ].iloc[0]
                )

                # ------------------------------------------------
                # Performance
                #
                # Black
                # ------------------------------------------------

                ax.bar(
                    i,
                    performance,
                    width=bar_width,
                    bottom=bottom,
                    color="black",
                    edgecolor="black",
                    linewidth=0.8,
                    zorder=3,
                )

                bottom += performance

                # ------------------------------------------------
                # Instability
                #
                # White with 45-degree hatching
                # ------------------------------------------------

                ax.bar(
                    i,
                    instability,
                    width=bar_width,
                    bottom=bottom,
                    color="white",
                    edgecolor="black",
                    hatch="////",
                    linewidth=0.8,
                    zorder=3,
                )

                bottom += instability

                # ------------------------------------------------
                # Metric separator
                #
                # Small horizontal line between metrics.
                # ------------------------------------------------

                ax.plot(
                    [
                        i - bar_width / 2,
                        i + bar_width / 2,
                    ],
                    [
                        bottom,
                        bottom,
                    ],
                    color="black",
                    linewidth=0.6,
                    zorder=4,
                )

        # ========================================================
        # X axis
        # ========================================================

        ax.set_xticks(
            range(
                len(stage2_values)
            )
        )

        ax.set_xticklabels(
            [
                f"{int(x):,}"
                for x in stage2_values
            ]
        )

        ax.set_xlabel(
            r"Stage-2 simulations ($n_{MC2}$)"
        )

        # ========================================================
        # Y axis
        # ========================================================

        ax.set_ylabel(
            "OOS performance + instability"
        )

        # ========================================================
        # Metric labels on the Y axis
        #
        # Because the four metrics have different numerical
        # magnitudes, put their names at the center of each
        # corresponding stacked section.
        # ========================================================

        # First calculate the average vertical position
        # of each metric across n_mc2.

        metric_centers = {
            metric: []
            for metric in metric_order
        }

        for stage2 in stage2_values:

            bottom = 0.0

            for metric in metric_order:

                row = df[
                    (
                        df[
                            "stage2_simulations"
                        ]
                        == stage2
                    )
                    & (
                        df["selection"]
                        == selection
                    )
                    & (
                        df["metric"]
                        == metric
                    )
                ]

                if row.empty:
                    continue

                performance = float(
                    row[
                        "oos_performance"
                    ].iloc[0]
                )

                instability = float(
                    row[
                        "oos_instability"
                    ].iloc[0]
                )

                total = (
                    performance
                    + instability
                )

                center = (
                    bottom
                    + total / 2
                )

                metric_centers[
                    metric
                ].append(
                    center
                )

                bottom += total

        # --------------------------------------------------------
        # Put labels at the mean center.
        # --------------------------------------------------------

        for metric in metric_order:

            if not metric_centers[metric]:
                continue

            center = np.mean(
                metric_centers[metric]
            )

            ax.text(
                -0.045,
                center,
                metric,
                transform=ax.get_yaxis_transform(),
                ha="right",
                va="center",
                fontsize=8,
                fontweight="bold",
            )

        # ========================================================
        # Grid
        # ========================================================

        ax.grid(
            axis="y",
            linestyle=":",
            linewidth=0.5,
            alpha=0.5,
            zorder=0,
        )

        ax.set_axisbelow(True)

        # ========================================================
        # Title
        # ========================================================

        ax.set_title(
            title,
            pad=12,
        )

    # ============================================================
    # Classical
    # ============================================================

    draw_panel(
        axes[0],
        "Classical",
        "(a) Classical",
    )

    # ============================================================
    # IDR
    # ============================================================

    draw_panel(
        axes[1],
        "IDR",
        "(b) IDR",
    )

    # ============================================================
    # Legend
    # ============================================================

    legend_handles = [
        Patch(
            facecolor="black",
            edgecolor="black",
            label="OOS performance",
        ),
        Patch(
            facecolor="white",
            edgecolor="black",
            hatch="////",
            label="OOS instability",
        ),
    ]

    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=2,
        frameon=False,
    )

    # ============================================================
    # Overall title
    # ============================================================

    fig.suptitle(
        (
            "OOS Performance and Instability Across "
            f"Stage-2 Simulations — "
            f"{strategy.replace('_', ' ').title()}"
        ),
        fontsize=12,
        y=1.055,
    )

    # ============================================================
    # Layout
    # ============================================================

    fig.subplots_adjust(
        left=0.18,
        right=0.98,
        bottom=0.12,
        top=0.86,
        wspace=0.12,
    )

    # ============================================================
    # Save
    # ============================================================

    fig.savefig(
        output_file,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)

    print(
        f"Saved: {output_file}"
    )

        # ============================================================
    # Print results table
    # ============================================================

    print("\n" + "=" * 90)
    print("OOS PERFORMANCE–INSTABILITY RESULTS")
    print("=" * 90)

    print(
        df.to_string(
            index=False,
            formatters={
                "oos_performance":
                    lambda x: f"{x:.6f}",
                "oos_instability":
                    lambda x: f"{x:.6f}",
            },
        )
    )

    print("=" * 90)

    return df

## extract_performance_degradation

In [6]:
def extract_performance_degradation(
    results_dir="../results",
    strategy="mean_reversion",
    data_start="2009-01-01",
    data_end="2023-12-29",
):
    """
    Extract compound excess return and calibration-to-OOS degradation
    for the best Classical and IDR configurations across all Stage-2
    Monte Carlo simulation levels.

    Returns one row per Stage-2 level with:

        stage2_simulations
        classical_compound_return
        idr_compound_return
        classical_degradation
        idr_degradation

    The values are extracted using the same selection logic as
    plot_performance_stability_tradeoff().
    """

    import json
    from pathlib import Path

    import numpy as np
    import pandas as pd

    results_dir = Path(results_dir)

    # ============================================================
    # Find ranking files
    # ============================================================

    classical_files = sorted(
        results_dir.glob(
            f"wf_{strategy}_*_uniq_classical_ranking.csv"
        )
    )

    idr_files = sorted(
        results_dir.glob(
            f"wf_{strategy}_*_uniq_idr_ranking.csv"
        )
    )

    if not classical_files:
        raise ValueError(
            f"No Classical ranking files found in {results_dir}"
        )

    if not idr_files:
        raise ValueError(
            f"No IDR ranking files found in {results_dir}"
        )

    # ============================================================
    # Match IDR files by experiment prefix
    # ============================================================

    idr_by_prefix = {}

    for f in idr_files:
        prefix = f.name.replace(
            "_uniq_idr_ranking.csv",
            "",
        )
        idr_by_prefix[prefix] = f

    records = []

    # ============================================================
    # Read experiments
    # ============================================================

    for classical_file in classical_files:

        prefix = classical_file.name.replace(
            "_uniq_classical_ranking.csv",
            "",
        )

        idr_file = idr_by_prefix.get(prefix)

        if idr_file is None:
            continue

        # --------------------------------------------------------
        # Read Stage-2 metadata
        # --------------------------------------------------------

        stage2 = None

        for json_file in results_dir.glob(
            f"{prefix}*.json"
        ):

            try:
                with open(
                    json_file,
                    "r",
                    encoding="utf-8",
                ) as f:
                    meta = json.load(f)

                stage2 = meta.get(
                    "N_SIMULATIONS_STAGE2",
                    meta.get(
                        "stage2_simulations",
                        meta.get("N_SIM_STAGE2"),
                    ),
                )

                if stage2 is not None:
                    break

            except Exception:
                continue

        if stage2 is None:
            continue

        try:
            stage2 = int(stage2)
        except Exception:
            continue

        # --------------------------------------------------------
        # Read ranking files
        # --------------------------------------------------------

        classical = pd.read_csv(
            classical_file
        )

        idr = pd.read_csv(
            idr_file
        )

        required = [
            "compound_excess_return",
            "degradation",
        ]

        if not all(
            col in classical.columns
            for col in required
        ):
            continue

        if not all(
            col in idr.columns
            for col in required
        ):
            continue

        if "classical_rank" not in classical.columns:
            continue

        if "idr_rank" not in idr.columns:
            continue

        # --------------------------------------------------------
        # Best Classical configuration
        # --------------------------------------------------------

        classical_rank = pd.to_numeric(
            classical["classical_rank"],
            errors="coerce",
        )

        classical_best = classical[
            np.isclose(
                classical_rank,
                1.0,
                equal_nan=False,
            )
        ]

        if classical_best.empty:
            continue

        classical_values = (
            classical_best[required]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .mean()
        )

        # --------------------------------------------------------
        # Best IDR configuration
        # --------------------------------------------------------

        idr_rank = pd.to_numeric(
            idr["idr_rank"],
            errors="coerce",
        )

        idr_best = idr[
            np.isclose(
                idr_rank,
                1.0,
                equal_nan=False,
            )
        ]

        if idr_best.empty:
            continue

        idr_values = (
            idr_best[required]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .mean()
        )

        # --------------------------------------------------------
        # Store
        # --------------------------------------------------------

        records.append(
            {
                "stage2_simulations": stage2,

                "classical_compound_return":
                    classical_values[
                        "compound_excess_return"
                    ],

                "idr_compound_return":
                    idr_values[
                        "compound_excess_return"
                    ],

                "classical_degradation":
                    classical_values[
                        "degradation"
                    ],

                "idr_degradation":
                    idr_values[
                        "degradation"
                    ],
            }
        )

    # ============================================================
    # Check results
    # ============================================================

    if not records:
        raise ValueError(
            "No matching Classical/IDR ranking pairs were found."
        )

    df = pd.DataFrame(records)

    # ============================================================
    # Average duplicate experiments at each Stage-2 level
    # ============================================================

    df = (
        df.groupby(
            "stage2_simulations",
            as_index=False,
        )
        .mean(numeric_only=True)
        .sort_values(
            "stage2_simulations"
        )
        .reset_index(drop=True)
    )

    # ============================================================
    # Print results
    # ============================================================

    print("\n" + "=" * 110)
    print(
        "COMPOUND EXCESS RETURN AND DEGRADATION"
    )
    print("=" * 110)

    print(
        df.to_string(
            index=False,
            formatters={
                "classical_compound_return":
                    lambda x: f"{x:.6f}",
                "idr_compound_return":
                    lambda x: f"{x:.6f}",
                "classical_degradation":
                    lambda x: f"{x:.6f}",
                "idr_degradation":
                    lambda x: f"{x:.6f}",
            },
        )
    )

    print("=" * 110)

    return df

## new

In [7]:
def extract_performance_degradation(
    results_dir="../results",
    strategy="mean_reversion",
    data_start="2009-01-01",
    data_end="2023-12-29",
):
    """
    Extract compound excess return and calibration-to-OOS degradation
    for all best Classical and IDR configurations across all Stage-2
    Monte Carlo simulation levels.

    Each rank-1 configuration is kept separately. No averaging is
    performed across methods or max_num_components.

    Returns one row per selected configuration with:

        stage2_simulations
        selection_method
        method
        max_num_components
        compound_excess_return
        degradation

    The values are extracted using the same selection logic as
    plot_performance_stability_tradeoff().
    """

    import json
    from pathlib import Path

    import numpy as np
    import pandas as pd

    results_dir = Path(results_dir)

    # ============================================================
    # Find ranking files
    # ============================================================

    classical_files = sorted(
        results_dir.glob(
            f"wf_{strategy}_*_uniq_classical_ranking.csv"
        )
    )

    idr_files = sorted(
        results_dir.glob(
            f"wf_{strategy}_*_uniq_idr_ranking.csv"
        )
    )

    if not classical_files:
        raise ValueError(
            f"No Classical ranking files found in {results_dir}"
        )

    if not idr_files:
        raise ValueError(
            f"No IDR ranking files found in {results_dir}"
        )

    # ============================================================
    # Match IDR files by experiment prefix
    # ============================================================

    idr_by_prefix = {}

    for f in idr_files:
        prefix = f.name.replace(
            "_uniq_idr_ranking.csv",
            "",
        )
        idr_by_prefix[prefix] = f

    records = []

    # ============================================================
    # Read experiments
    # ============================================================

    for classical_file in classical_files:

        prefix = classical_file.name.replace(
            "_uniq_classical_ranking.csv",
            "",
        )

        idr_file = idr_by_prefix.get(prefix)

        if idr_file is None:
            continue

        # --------------------------------------------------------
        # Read Stage-2 metadata
        # --------------------------------------------------------

        stage2 = None

        for json_file in results_dir.glob(
            f"{prefix}*.json"
        ):

            try:
                with open(
                    json_file,
                    "r",
                    encoding="utf-8",
                ) as f:
                    meta = json.load(f)

                stage2 = meta.get(
                    "N_SIMULATIONS_STAGE2",
                    meta.get(
                        "stage2_simulations",
                        meta.get("N_SIM_STAGE2"),
                    ),
                )

                if stage2 is not None:
                    break

            except Exception:
                continue

        if stage2 is None:
            continue

        try:
            stage2 = int(stage2)
        except Exception:
            continue

        # --------------------------------------------------------
        # Read ranking files
        # --------------------------------------------------------

        classical = pd.read_csv(
            classical_file
        )

        idr = pd.read_csv(
            idr_file
        )

        required = [
            "compound_excess_return",
            "degradation",
        ]

        if not all(
            col in classical.columns
            for col in required
        ):
            continue

        if not all(
            col in idr.columns
            for col in required
        ):
            continue

        if "classical_rank" not in classical.columns:
            continue

        if "idr_rank" not in idr.columns:
            continue

        if "method" not in classical.columns:
            continue

        if "max_num_components" not in classical.columns:
            continue

        if "method" not in idr.columns:
            continue

        if "max_num_components" not in idr.columns:
            continue

        # ========================================================
        # Best Classical configurations
        # ========================================================

        classical_rank = pd.to_numeric(
            classical["classical_rank"],
            errors="coerce",
        )

        classical_best = classical[
            np.isclose(
                classical_rank,
                1.0,
                equal_nan=False,
            )
        ].copy()

        for _, row in classical_best.iterrows():

            records.append(
                {
                    "stage2_simulations": stage2,
                    "selection_method": "Classical",
                    "method": row["method"],
                    "max_num_components":
                        row["max_num_components"],
                    "compound_excess_return":
                        row["compound_excess_return"],
                    "degradation":
                        row["degradation"],
                }
            )

        # ========================================================
        # Best IDR configurations
        # ========================================================

        idr_rank = pd.to_numeric(
            idr["idr_rank"],
            errors="coerce",
        )

        idr_best = idr[
            np.isclose(
                idr_rank,
                1.0,
                equal_nan=False,
            )
        ].copy()

        for _, row in idr_best.iterrows():

            records.append(
                {
                    "stage2_simulations": stage2,
                    "selection_method":
                        "Instability–Degradation",
                    "method": row["method"],
                    "max_num_components":
                        row["max_num_components"],
                    "compound_excess_return":
                        row["compound_excess_return"],
                    "degradation":
                        row["degradation"],
                }
            )

    # ============================================================
    # Check results
    # ============================================================

    if not records:
        raise ValueError(
            "No matching Classical/IDR rank-1 configurations were found."
        )

    df = pd.DataFrame(records)

    # ============================================================
    # Convert numeric columns
    # ============================================================

    df["stage2_simulations"] = pd.to_numeric(
        df["stage2_simulations"],
        errors="coerce",
    )

    df["max_num_components"] = pd.to_numeric(
        df["max_num_components"],
        errors="coerce",
    )

    df["compound_excess_return"] = pd.to_numeric(
        df["compound_excess_return"],
        errors="coerce",
    )

    df["degradation"] = pd.to_numeric(
        df["degradation"],
        errors="coerce",
    )

    # ============================================================
    # Sort
    # ============================================================

    selection_order = {
        "Classical": 0,
        "Instability–Degradation": 1,
    }

    df["_selection_order"] = (
        df["selection_method"]
        .map(selection_order)
    )

    df = (
        df.sort_values(
            [
                "stage2_simulations",
                "_selection_order",
                "method",
                "max_num_components",
            ]
        )
        .drop(
            columns="_selection_order"
        )
        .reset_index(drop=True)
    )

    # ============================================================
    # Print results
    # ============================================================

    print("\n" + "=" * 120)
    print(
        "COMPOUND EXCESS RETURN AND CALIBRATION-TO-OOS DEGRADATION"
    )
    print("=" * 120)

    print(
        df.to_string(
            index=False,
            formatters={
                "compound_excess_return":
                    lambda x: f"{x:.6f}",
                "degradation":
                    lambda x: f"{x:.6f}",
            },
        )
    )

    print("=" * 120)

    return df

# RESEARCH

In [ ]:
# Execution order in MPF folder

# 1. python scripts/run_walk_forward_optimization.py

# 2. python scripts/evaluate_train_results.py 

<table>
  <tr>
    <th>Workflow</th>
    <th>Train</th>
    <th>Test</th>
  </tr>
  <tr>
    <td>1. <code>run_walk_forward_optimization.py</code></td>
    <td><b>Step 1</b><br>N2_sim ∈ {100, 250, 500, 1000, 1500}<br>for all {q, n_max_components}</td>
    <td><b>Step 3</b><br>Apply N2_sim* for<br>{q*, n_max_components*} from Step 2</td>
  </tr>
  <tr>
    <td>2. <code>evaluate_train_results.py</code></td>
    <td><b>Step 2</b><br>4 best Classic + 4 best IDR</td>
    <td><b>Step 4</b><br>Compare results vs Step 2</td>
  </tr>
</table>

In [ ]:
                 MC2 = 100
                       │
        ┌──────────────┴──────────────┐
        │                             │
   Classical                         IDR
        │                             │
   ┌────┼────┬────┐             ┌────┼────┬────┐
 Sharpe Calmar R/R Robust      Sharpe Calmar R/R Robust
   │      │     │     │           │      │     │     │
 best    best  best  best        best   best  best  best

## Experiments summary

In [9]:
classical_csv = pd.read_csv("../results/wf_mean_reversion_cal6_move12_test12_ncomp5-20_20260903_140908_8d95d5eb78df_uniq_classical_ranking.csv")
classical_csv.columns

Index(['method', 'max_num_components', 'P_positive', 'mean_excess_return',
       'compound_excess_return', 'sharpe_excess', 'threshold_instability',
       'buy_thr_mean_abs_change', 'sell_thr_mean_abs_change',
       'oos_instability', 'calibration_return_mean', 'oos_return_mean',
       'degradation', 'n_windows', 'rank_P_positive', 'rank_mean_excess',
       'rank_compound_excess', 'rank_sharpe_excess', 'classical_average_rank',
       'classical_rank'],
      dtype='object')

In [10]:
idr_csv = pd.read_csv("../results/wf_mean_reversion_cal6_move12_test12_ncomp5-20_20260903_140908_8d95d5eb78df_uniq_idr_ranking.csv")
idr_csv.columns

Index(['method', 'max_num_components', 'P_positive', 'mean_excess_return',
       'compound_excess_return', 'sharpe_excess', 'threshold_instability',
       'buy_thr_mean_abs_change', 'sell_thr_mean_abs_change',
       'oos_instability', 'calibration_return_mean', 'oos_return_mean',
       'degradation', 'n_windows', 'rank_P_positive', 'rank_mean_excess',
       'rank_compound_excess', 'rank_sharpe_excess', 'classical_average_rank',
       'classical_rank', 'rank_threshold_instability', 'rank_oos_instability',
       'rank_degradation', 'rank_oos_performance', 'idr_score', 'idr_rank'],
      dtype='object')

In [ ]:
mom_100 = pd.read_csv("../results/wf_momentum_cal6_move12_test12_ncomp5-20_20260914_071245_6d55493c74df.csv")

In [8]:
summary = summarize_mpf_results("../results")

print("\n" + "=" * 160)
print("MPF — EXPERIMENT SUMMARY")
print("=" * 160)

print(
    summary[
        [
            "status",
            "run_date",
            "start_time",
            "end_time",
            "duration",
            "strategy",
            "start_date",
            "end_date",
            "calibration",
            "move",
            "test",
            "components",
            "stage1_simulations",
            "stage2_simulations",
            "classical_ranking",
            "idr_ranking",
            "final_selection",
        ]
    ].to_string(index=False)
)

print("=" * 160)


MPF — EXPERIMENT SUMMARY
   status   run_date start_time end_time duration       strategy start_date   end_date  calibration  move  test components  stage1_simulations  stage2_simulations  classical_ranking  idr_ranking  final_selection
  RUNNING 2026-09-15   15:41:00     None 14:58:04       momentum 2009-01-01        NaT            6    12    12       5–20                 500                1000              False        False            False
COMPLETED 2026-09-15   07:01:09 14:36:47 07:35:38       momentum 2009-01-01 2023-12-29            6    12    12       5–20                 500                 500              False        False            False
COMPLETED 2026-09-14   13:37:48 06:44:17 17:06:29       momentum 2009-01-01 2023-12-29            6    12    12       5–20                 500                 250              False        False            False
COMPLETED 2026-09-14   07:12:45 11:19:48 04:07:03       momentum 2009-01-01 2023-12-29            6    12    12       5–20    

In [ ]:
results = plot_selection_stability(
    results_dir="../results",
    strategy="mean_reversion",
    data_end="2023-12-29"
)

Created: article_selection_stability_mean_reversion_2009-01-01_2023-12-29.pdf

SELECTION STABILITY RESULTS — Mean Reversion (2009-01-01 to 2023-12-29)
 stage2_simulations        selection_method threshold_instability oos_instability selected_portfolio_size
                100               Classical              0.105087        0.162565                   13.75
                100 Instability–Degradation              0.051119        0.130259                    7.75
                250               Classical              0.085757        0.141244                   15.50
                250 Instability–Degradation              0.030288        0.123095                   14.50
                500               Classical              0.079520        0.149318                   11.25
                500 Instability–Degradation              0.035729        0.094291                   14.00
               1000               Classical              0.089881        0.131016                   15.25
 

In [5]:
results = plot_classical_performance_comparison(
    results_dir="../results",
    strategy="mean_reversion",
    data_start="2009-01-01",
    data_end="2023-12-29"
)

Created: article_classical_performance_mean_reversion_2009-01-01_2023-12-29.pdf

CLASSICAL PERFORMANCE COMPARISON RESULTS — Mean Reversion (2009-01-01 to 2023-12-29)
 stage2_simulations        selection_method P_positive mean_excess_return compound_excess_return sharpe_excess selected_portfolio_size
                100               Classical   0.666667           0.060195               0.506390      0.331450                   15.00
                100 Instability–Degradation   0.444444           0.017942               0.133121      0.180844                    7.00
                250               Classical   0.666667           0.040914               0.317280      0.260532                   16.00
                250 Instability–Degradation   0.333333          -0.011324              -0.120590     -0.140006                   20.00
                500               Classical   0.666667           0.057709               0.534243      0.395862                    7.00
                500 Inst

In [22]:
results = plot_performance_stability_tradeoff(
    results_dir="../results",
    strategy="mean_reversion",
    data_start="2009-01-01",
    data_end="2023-12-29"
)

Found 5 Classical ranking files and 5 IDR ranking files.
Saved: ../results/performance_stability_tradeoff_mean_reversion_2009-01-01_2023-12-29.pdf

PERFORMANCE–STABILITY–DEGRADATION TRADE-OFF RESULTS
 stage2_simulations classical_compound_return idr_compound_return classical_threshold_instability idr_threshold_instability classical_oos_instability idr_oos_instability classical_degradation idr_degradation
                100                  0.506390            0.133121                        0.112618                  0.057517                  0.181612            0.099211              3.487427        2.064695
                250                  0.317280           -0.120590                        0.023252                  0.033842                  0.157040            0.080883              2.120001        1.854921
                500                  0.534243           -0.233516                        0.055751                  0.022526                  0.145782            0.066826       

In [19]:
results = plot_oos_performance_instability_stacked(
    results_dir="../results",
    strategy="mean_reversion",
    data_start="2009-01-01",
    data_end="2023-12-29",
)

Found 5 Classical ranking files and 5 IDR ranking files.

Methods found:
['Calmar', 'Return/Risk', 'Robust', 'Sharpe']
Saved: ../results/oos_performance_instability_stacked_mean_reversion_2009-01-01_2023-12-29.pdf

OOS PERFORMANCE–INSTABILITY RESULTS
 stage2_simulations selection      metric oos_performance oos_instability
                100 Classical      Calmar        0.087939        0.164010
                100 Classical Return/Risk        0.085005        0.159936
                100 Classical      Robust        0.064465        0.136129
                100 Classical      Sharpe        0.093622        0.159791
                100       IDR      Calmar        0.087939        0.164010
                100       IDR Return/Risk        0.085005        0.159936
                100       IDR      Robust        0.064465        0.136129
                100       IDR      Sharpe        0.093622        0.159791
                250 Classical      Calmar        0.088505        0.142748
         

In [28]:
df = extract_performance_degradation(
    results_dir="../results",
    strategy="mean_reversion",
    data_start="2009-01-01",
    data_end="2023-12-29",
)

df


COMPOUND EXCESS RETURN AND CALIBRATION-TO-OOS DEGRADATION
 stage2_simulations        selection_method      method  max_num_components compound_excess_return degradation
                100               Classical      Calmar                  15               0.506390    3.487427
                100 Instability–Degradation      Robust                   7               0.133121    2.064695
                250               Classical      Robust                  16               0.317280    2.120001
                250 Instability–Degradation      Robust                  20              -0.120590    1.854921
                500               Classical      Calmar                   7               0.534243    3.615921
                500 Instability–Degradation      Robust                  19              -0.233516    1.952277
               1000               Classical Return/Risk                  14               0.453914    4.865309
               1000 Instability–Degradation      Robu

,stage2_simulations,selection_method,method,max_num_components,compound_excess_return,degradation
0,100,Classical,Calmar,15,0.506390,3.487427
1,100,Instability–Degradation,Robust,7,0.133121,2.064695
2,250,Classical,Robust,16,0.317280,2.120001
3,250,Instability–Degradation,Robust,20,-0.120590,1.854921
4,500,Classical,Calmar,7,0.534243,3.615921
5,500,Instability–Degradation,Robust,19,-0.233516,1.952277
6,1000,Classical,Return/Risk,14,0.453914,4.865309
7,1000,Instability–Degradation,Robust,18,0.275052,1.871331
8,1500,Classical,Return/Risk,9,0.576670,4.273072
9,1500,Instability–Degradation,Robust,18,-0.162305,2.000131
